# Tái lập kiểm tra ROI Site04

Mục tiêu: người khác có thể tái tạo mask tâm pixel và so sánh năm raster với evidence đã lưu. Điểm minh họa không phải địa điểm hạ cánh. Kết quả không xác nhận vertical datum, frame identity hoặc horizon.

Yêu cầu: Python, Jupyter, NumPy, pandas, Matplotlib, GeoPandas, PyProj, Shapely và Rasterio có hỗ trợ `opener` (đã chạy với 1.5.2). Chạy tại thư mục repo hoặc notebook. Mặc định `FETCH_REMOTE=False`: cần cache local, không tải mạng. Để tái lập trên máy khác đặt biến môi trường `SITE04_FETCH_REMOTE=1` trước khi chạy: tải archive ROI nhỏ và đọc HTTP Range có giới hạn 24 MiB/lớp, không tải toàn raster. Cache trong `data/preview/site04_roi_reproduction/` bị Git bỏ qua. Mỗi nguồn có thể thay đổi: sự khác biệt hash phải được review, không ghi đè evidence cũ.

In [1]:
import os,io,time,json,hashlib,urllib.request,tarfile,tempfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from pyproj import CRS
from shapely.geometry import mapping,Point
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'docs/site04-roi-coverage.md').exists())
CACHE=ROOT/'data/preview/site04_roi_reproduction'; CACHE.mkdir(parents=True,exist_ok=True)
FETCH_REMOTE=os.environ.get('SITE04_FETCH_REMOTE','0')=='1'
EXPECTED=json.loads((ROOT/'notebooks/evidence/site04-roi-coverage.json').read_text())
EXPECTED_ROI=EXPECTED['roi_sha256']
ROI_URL='https://pgda.gsfc.nasa.gov/data/LOLA_5mpp/Site04/Site04_ROIs.tgz'
ROI_FILE=CACHE/'Site04_ROIs.tgz'
if not ROI_FILE.exists():
    if not FETCH_REMOTE: raise FileNotFoundError('ROI cache missing: set SITE04_FETCH_REMOTE=1 to fetch the small archive')
    req=urllib.request.Request(ROI_URL,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req,timeout=30) as response: payload=response.read(1024*1024+1)
    if len(payload)>1024*1024: raise ValueError('ROI archive exceeds 1 MiB limit')
    if hashlib.sha256(payload).hexdigest()!=EXPECTED_ROI: raise ValueError('ROI archive changed; review source before updating evidence')
    ROI_FILE.write_bytes(payload)
assert hashlib.sha256(ROI_FILE.read_bytes()).hexdigest()==EXPECTED_ROI, 'ROI archive hash changed'
# Select files by exact basenames; do not extract arbitrary archive paths.
with tempfile.TemporaryDirectory() as td:
    with tarfile.open(ROI_FILE,'r:gz') as tar:
        for ext in ('shp','shx','dbf','prj'):
            members=[m for m in tar.getmembers() if m.isfile() and Path(m.name).name==f'Site04_ROIs.{ext}']
            if len(members)!=1 or members[0].size>1_000_000: raise ValueError(f'Unexpected ROI member: {ext}')
            (Path(td)/f'Site04_ROIs.{ext}').write_bytes(tar.extractfile(members[0]).read())
    roi=gpd.read_file(Path(td)/'Site04_ROIs.shp')
assert len(roi)==4
class RangeFile(io.RawIOBase):
    """Seekable read-only source: validate 206 bodies; reject whole-file fallback."""
    def __init__(self,url,size,blocks,traffic):
        super().__init__(); self.url=url; self.size=size; self.blocks=blocks; self.traffic=traffic; self.pos=0
        self.block_size=512*1024
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self,offset,whence=0):
        self.pos=int(offset if whence==0 else self.pos+offset if whence==1 else self.size+offset)
        if self.pos<0: raise ValueError('Negative offset')
        return self.pos
    def read(self,n=-1):
        if n<0: raise ValueError('Unbounded source read disabled')
        end=min(self.size,self.pos+n); chunks=[]
        while self.pos<end:
            idx=self.pos//self.block_size; start=idx*self.block_size; stop=min(start+self.block_size,self.size)-1
            if idx not in self.blocks:
                last=None
                for attempt in range(3):
                    if self.traffic['body_bytes']+(stop-start+1)>24*1024**2:
                        raise IOError('Per-site HTTP body budget exceeded (24 MiB including retries)')
                    try:
                        req=urllib.request.Request(self.url,headers={'User-Agent':'Mozilla/5.0','Range':f'bytes={start}-{stop}'})
                        with urllib.request.urlopen(req,timeout=25) as r:
                            if r.status!=206 or r.headers.get('Content-Range')!=f'bytes {start}-{stop}/{self.size}':
                                raise IOError('Source ignored/misreported requested byte range')
                            body=r.read(stop-start+2)
                            self.traffic['body_bytes']+=len(body)
                            if len(body)!=stop-start+1: raise IOError('Truncated or oversized byte range')
                            self.blocks[idx]=body; self.traffic['ranges'].append([start,stop]); break
                    except Exception as exc:
                        last=exc
                        if attempt==2: raise IOError(f'Failed range {start}-{stop}: {exc}') from exc
                        time.sleep(1)
            data=self.blocks[idx]; take=min(end-self.pos,stop-self.pos+1)
            chunks.append(data[self.pos-start:self.pos-start+take]); self.pos+=take
        return b''.join(chunks)
    def readinto(self,b):
        d=self.read(len(b)); b[:len(d)]=d; return len(d)



BASE='https://pgda.gsfc.nasa.gov/data/LOLA_5mpp/Site04/Site04_final_adj_5mpp_'
FILES={'surface':'surf','count':'ldec','slope':'slp','z_rms':'toterr','slope_rms':'slperr'}
refs={x['layer']:x for x in EXPECTED['layers']}
def load_layer(item):
    name,suffix=item; path=CACHE/f'{name}.npz'; ref=refs[name]
    if path.exists():
        with np.load(path,allow_pickle=False) as f: a=f['data'];mask=f['roi_mask'];meta=json.loads(str(f['metadata']))
    else:
        if not FETCH_REMOTE: raise FileNotFoundError(f'{name} cache missing: set SITE04_FETCH_REMOTE=1')
        url=BASE+suffix+'.tif'
        with urllib.request.urlopen(urllib.request.Request(url,method='HEAD',headers={'User-Agent':'Mozilla/5.0'}),timeout=30) as h:
            headers={k:h.headers.get(k) for k in ('Content-Length','ETag','Last-Modified','Content-Type')}
        if 'tiff' not in str(headers['Content-Type']).lower(): raise ValueError('Unexpected source content')
        blocks={}; traffic={'body_bytes':0,'ranges':[]}
        def opener(path,mode='rb'):
            if str(path)!=url: raise FileNotFoundError(path)
            return RangeFile(url,int(headers['Content-Length']),blocks,traffic)
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(url,opener=opener) as ds:
                # Matching projection parameters is an interpretation, not authoritative frame equality.
                if CRS.from_user_input(roi.crs).to_proj4()!=CRS.from_user_input(ds.crs).to_proj4(): raise ValueError('Projection parameters differ')
                if not np.allclose(ds.res,(5,5)): raise ValueError('Unexpected resolution')
                w=from_bounds(*roi.total_bounds,transform=ds.transform).round_offsets().round_lengths(); w=rasterio.windows.Window(w.col_off-2,w.row_off-2,w.width+4,w.height+4)
                a=ds.read(1,window=w,masked=True).astype('float32').filled(np.nan)*ds.scales[0]+ds.offsets[0]
                tr=ds.window_transform(w); mask=geometry_mask([mapping(x) for x in roi.geometry],out_shape=a.shape,transform=tr,invert=True,all_touched=False)
                meta={'window':[w.col_off,w.row_off,w.width,w.height],'window_transform':list(tr)[:6],'source_headers':headers,'http_body_bytes':traffic['body_bytes']}
        np.savez_compressed(path,data=a,roi_mask=mask,metadata=json.dumps(meta))
    h=hashlib.sha256(np.ascontiguousarray(a,dtype='<f4').tobytes()).hexdigest()
    # Fail closed on source/grid changes; leave the old evidence untouched.
    if h!=ref['window_sha256'] or meta['window']!=ref['window'] or meta['window_transform']!=ref['window_transform']: raise ValueError(f'{name}: window/source differs from reviewed snapshot')
    from rasterio.transform import Affine
    check=geometry_mask([mapping(x) for x in roi.geometry],out_shape=a.shape,transform=Affine(*meta['window_transform']),invert=True,all_touched=False)
    if not np.array_equal(mask,check): raise ValueError(f'{name}: ROI mask mismatch')
    valid=mask & np.isfinite(a);v=a[valid]
    return name,a,mask,meta,{'layer':name,'roi_cells':int(mask.sum()),'valid_cells':int(valid.sum()),'min':float(v.min()),'max':float(v.max()),'median':float(np.median(v))}
with ThreadPoolExecutor(max_workers=5) as pool: results=list(pool.map(load_layer,FILES.items()))
arrays={name:a for name,a,mask,meta,stats in results}; masks={name:mask for name,a,mask,meta,stats in results}
summary=pd.DataFrame([row for name,a,mask,meta,row in results]); display(summary)
assert all(row['roi_cells']==526 and row['valid_cells']==526 for *_,row in results)
mask=masks['count']; zero=int(((arrays['count']==0)&mask).sum()); positive=int(((arrays['count']>0)&mask).sum())
assert (zero,positive)==(468,58)
print('Count 0:',zero,'; count > 0:',positive,'; zeros remain valid data.')


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-i

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


,layer,roi_cells,valid_cells,min,max,median
0,surface,526,526,1723.327881,1739.121826,1732.627686
1,count,526,526,0.000000,2.000000,0.000000
2,slope,526,526,0.052339,10.325006,2.860505
3,z_rms,526,526,0.088361,0.962349,0.344611
4,slope_rms,526,526,0.643458,4.817510,1.613893


Count 0: 468 ; count > 0: 58 ; zeros remain valid data.


## Kiểm tra điểm minh họa

Chọn tâm pixel có LDEC > 0 xa biên union nhất, khoảng cách đo trên mặt phẳng chiếu. Quy tắc này chỉ tạo ví dụ tái lập, không chứng minh an toàn hạ cánh. CRS `.equals()` giữa shapefile/raster là false dù tham số PROJ trùng; vẫn cần xác thực frame identity từ nguồn.

In [2]:
from rasterio.transform import Affine
from pyproj import Transformer
m=refs['surface']; t=Affine(*m['window_transform']); union=roi.geometry.union_all(); candidates=[]
for i,j in np.argwhere(masks['count'] & (arrays['count']>0)):
    x,y=t*(float(j)+.5,float(i)+.5); pt=Point(x,y); candidates.append((pt.distance(union.boundary),x,y,int(i),int(j)))
d,x,y,i,j=max(candidates)
assert np.allclose([d,x,y],[15,-2652.5,-5952.5])
lon,lat=Transformer.from_crs(m['crs_wkt'],'+proj=longlat +a=1737400 +b=1737400 +no_defs',always_xy=True).transform(x,y)
print(f'Điểm minh họa: lat {lat:.8f}, lon {lon:.8f}; margin {d:.1f} m')
print({name:float(arr[i,j]) for name,arr in arrays.items()})
print('Điểm kiểm thử; không dùng như điểm hạ cánh hoặc observer đã phê duyệt.')

Điểm minh họa: lat -89.78509166, lon -155.98171723; margin 15.0 m
{'surface': 1737.1783447265625, 'count': 1.0, 'slope': 2.2839622497558594, 'z_rms': 0.1362612098455429, 'slope_rms': 1.0714844465255737}
Điểm kiểm thử; không dùng như điểm hạ cánh hoặc observer đã phê duyệt.


Nếu hash không khớp, giữ bản evidence cũ và so sánh ETag, thời gian, phiên bản và nguồn trước khi cập nhật. Vertical datum PGDA 78 và chuyển đổi AGL/ellipsoid vẫn cần xác nhận trực tiếp; kết quả này không mở khóa Horizon đã xác minh.